# Dalya — YASA Spindle Features (68 subjects, fixed A–E)

Cleaned from the original v5 notebook. Original remains unchanged. **Default mode verifies saved predictions; does not retrain or re-extract EEG.**

Original: `Dalya_SleepEDF_TC1_C1_YASA_Full_AE_v5 (1).ipynb`. Holdout subjects must remain untouched.

## 1. Session setup
Run these cells in order. These are the original pinned team setup functions, preserved without changing feature engineering or folds.

### Setup 3 — original source cell

In [ ]:
# Session paths and options.
from pathlib import Path
import errno
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import pickle
import platform
import shutil
import subprocess
import sys
import time

COURSE_REVISION = "5949b723bbd34d172f072446dfcb802ef54b49f7"
COURSE_URL = "https://github.com/farhad-abtahi/CM2013.git"
DATA = "/content/drive/MyDrive/CM2013_data"
TRAIN_DIR = Path(DATA) / "train"
CACHE_VIGGY = str(Path(DATA) / "cache" / "viggy")
EXPERIMENTS_DIR = Path(DATA) / "experiments"
LOCAL_SESSION_DIR = Path("/content/CM2013_viggy_runtime")
COURSE_DIR = Path("/content/_bsp_repo")

# Optional training: False skips preparation and fitting for that variant.
RUN_TC1_68 = False
RUN_TC1_C1_68 = False
RUN_TC1_LP40_68 = False
RUN_SEL40_68 = False

# Optional diagnostics may read recordings; they do not fit models.
RUN_DIAGNOSTICO_SINAL_68 = False
RUN_DIAGNOSTICO_SEL40_68 = False

# Plot saved predictions; None skips the plots.
MOSTRAR_MATRIZES_68 = None
USE_REAL = True
BASELINE_CONFIG = {}
SESSAO_PRONTA = False
_SETUP_STAGES_68 = set()
_IO_RETRIES = 3
_DRIVE_REMOUNT_USED = False
_DRIVER_RECOVERIES = []

EXPECTED_FOLDS_68 = {
    "A": "SC400 SC405 SC411 SC417 SC422 SC428 SC434 SC441 SC446 SC452 SC458 SC463 SC471 SC477".split(),
    "B": "SC401 SC406 SC412 SC418 SC423 SC429 SC435 SC442 SC447 SC453 SC459 SC465 SC473 SC481".split(),
    "C": "SC402 SC407 SC413 SC419 SC425 SC430 SC436 SC443 SC449 SC454 SC460 SC466 SC474 SC482".split(),
    "D": "SC403 SC409 SC414 SC420 SC426 SC431 SC437 SC444 SC450 SC455 SC461 SC467 SC475".split(),
    "E": "SC404 SC410 SC415 SC421 SC427 SC433 SC438 SC445 SC451 SC457 SC462 SC470 SC476".split(),
}
HOLDOUT_GROUPS = set(
    "SC408 SC416 SC424 SC432 SC440 SC448 SC456 SC464 SC472 SC480".split()
)
LOCAL_SESSION_DIR.mkdir(parents=True, exist_ok=True)

def _require_stage_68(nome):
    if nome not in _SETUP_STAGES_68:
        raise RuntimeError(
            f"Setup incomplete ({nome}). Run section 1 in order."
        )

print("Data folder:", DATA)
ativos = [
    nome for nome, ligado in {
        "TC1": RUN_TC1_68, "TC1 + corte": RUN_TC1_C1_68,
        "TC1 + LP40": RUN_TC1_LP40_68, "SEL40": RUN_SEL40_68,
    }.items() if ligado
]
print("Enabled training runs:", ", ".join(ativos) or "none")


### Setup 5 — original source cell

In [ ]:
from google.colab import drive
import stat as _stat

drive.mount("/content/drive")

def _drive_io(operacao, descricao):
    """Retry transient I/O errors only, with at most one remount."""
    global _DRIVE_REMOUNT_USED
    transitorios = {
        errno.EIO, errno.ECONNABORTED, errno.ECONNRESET,
        errno.ENOTCONN, errno.ETIMEDOUT, getattr(errno, "ESTALE", 116),
    }
    for tentativa in range(_IO_RETRIES):
        try:
            return operacao()
        except OSError as exc:
            if exc.errno not in transitorios or tentativa == _IO_RETRIES - 1:
                raise OSError(
                    exc.errno,
                    f"{descricao}: {exc.strerror or str(exc)}. "
                    "Setup stopped. Existing data and results have been preserved.",
                    getattr(exc, "filename", None),
                ) from exc
            print(
                f"Temporary Drive error: {descricao}. "
                f"Retry {tentativa + 2}/{_IO_RETRIES}.",
                flush=True,
            )
            if exc.errno in {errno.ECONNABORTED, errno.ENOTCONN, errno.ECONNRESET}:
                if not _DRIVE_REMOUNT_USED:
                    _DRIVE_REMOUNT_USED = True
                    _DRIVER_RECOVERIES.append(descricao)
                    drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def _file_signature(caminho):
    caminho = Path(caminho)
    st = _drive_io(caminho.stat, f"check {caminho.name}")
    return {"name": caminho.name, "bytes": st.st_size, "mtime_ns": st.st_mtime_ns}

def _drive_is_file(caminho):
    caminho = Path(caminho)
    try:
        st = _drive_io(caminho.stat, f"check {caminho.name}")
    except FileNotFoundError:
        return False
    return _stat.S_ISREG(st.st_mode)

def _local_copy(caminho, grupo="files"):
    """Temporary Colab copy. The source on Drive is read-only."""
    caminho = Path(caminho)
    assinatura = _file_signature(caminho)
    chave = hashlib.sha256(
        json.dumps([str(caminho), assinatura], sort_keys=True).encode()
    ).hexdigest()[:20]
    destino = LOCAL_SESSION_DIR / grupo / f"{chave}_{caminho.name}"
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.is_file() and destino.stat().st_size == assinatura["bytes"]:
        return destino

    def copiar():
        temporario = destino.with_suffix(destino.suffix + ".tmp")
        shutil.copyfile(caminho, temporario)
        assert temporario.stat().st_size == assinatura["bytes"], (
            f"Incomplete copy: {caminho.name}"
        )
        assert _file_signature(caminho) == assinatura, (
            f"The file changed while it was being read: {caminho.name}"
        )
        temporario.replace(destino)
        return destino

    return _drive_io(copiar, f"copy {caminho.name} to local disk")

def _read_json(caminho):
    local = _local_copy(caminho, "reports")
    return json.loads(local.read_text(encoding="utf-8"))

def _atomic_bytes(caminho, conteudo):
    caminho = Path(caminho)
    def escrever():
        caminho.parent.mkdir(parents=True, exist_ok=True)
        temporario = caminho.with_suffix(caminho.suffix + ".tmp")
        temporario.write_bytes(conteudo)
        temporario.replace(caminho)
    _drive_io(escrever, f"save {caminho.name}")

def converter_numpy(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    raise TypeError(f"Unsupported type: {type(obj).__name__}")

def _atomic_json(caminho, objeto):
    conteudo = json.dumps(
        objeto, default=converter_numpy, ensure_ascii=False, indent=2
    ).encode("utf-8")
    _atomic_bytes(caminho, conteudo)

assert _stat.S_ISDIR(_drive_io(Path(DATA).stat, "open the data folder").st_mode), DATA
assert _stat.S_ISDIR(_drive_io(TRAIN_DIR.stat, "open the training folder").st_mode), TRAIN_DIR
print("Drive is mounted and the data folder is accessible.")
_SETUP_STAGES_68.add("drive")


### Setup 7 — original source cell

In [ ]:
# Read the saved configuration before importing scientific packages.
_require_stage_68("drive")
ref_a_path = EXPERIMENTS_DIR / "F2_68_fixed_AE_fold_A.json"
ref_a_guardada_68 = _read_json(ref_a_path)
config_f2_guardada_68 = ref_a_guardada_68.get("metadata", ref_a_guardada_68)
assert config_f2_guardada_68["course_revision"] == COURSE_REVISION
assert config_f2_guardada_68["fold_assignment"] == EXPECTED_FOLDS_68

# Reuse or fetch the course code at the pinned revision.
if not (COURSE_DIR / ".git").is_dir():
    if COURSE_DIR.exists() and any(COURSE_DIR.iterdir()):
        raise RuntimeError(
            f"{COURSE_DIR} already contains files. Use a different folder for the course code."
        )
    COURSE_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "--quiet", str(COURSE_DIR)], check=True)
    subprocess.run(
        ["git", "-C", str(COURSE_DIR), "remote", "add", "origin", COURSE_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(COURSE_DIR), "fetch", "--quiet", "--depth", "1",
         "origin", COURSE_REVISION], check=True,
    )
    subprocess.run(
        ["git", "-C", str(COURSE_DIR), "checkout", "--quiet",
         "--detach", "FETCH_HEAD"], check=True,
    )

course_revision_loaded = subprocess.check_output(
    ["git", "-C", str(COURSE_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert course_revision_loaded == COURSE_REVISION, (
    "The existing checkout is at a different revision. Keep your changes "
    "and use a fresh runtime or a separate folder for the course code."
)

# Match the package versions recorded for F2.
PACKAGE_MAP = {
    "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
    "mne": "mne", "matplotlib": "matplotlib", "pandas": "pandas",
    "imblearn": "imbalanced-learn",
}
reference_runtime = config_f2_guardada_68["runtime"]
instalar = []
for module, distribution in PACKAGE_MAP.items():
    expected = reference_runtime.get(distribution)
    try:
        installed = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if expected and expected != "not installed" and installed != expected:
        instalar.append(f"{distribution}=={expected}")
    elif installed is None:
        instalar.append(distribution)

if instalar:
    print("Aligning package versions:", ", ".join(instalar), flush=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *instalar],
        check=True,
    )

# Reinstalling a package does not replace an already imported module.
loaded_conflicts = {}
for module, distribution in PACKAGE_MAP.items():
    expected = reference_runtime.get(distribution)
    loaded = sys.modules.get(module)
    if expected and loaded is not None and hasattr(loaded, "__version__"):
        if loaded.__version__ != expected:
            loaded_conflicts[distribution] = {
                "loaded": loaded.__version__, "required": expected
            }
if loaded_conflicts:
    raise RuntimeError(
        "The packages were aligned, but the runtime still has old versions loaded. "
        "Restart the Colab runtime and choose Run all again. "
        + str(loaded_conflicts)
    )

for subfolder in ("tracks", "src", "src/bsp"):
    entry = str(COURSE_DIR / subfolder)
    if entry not in sys.path:
        sys.path.insert(0, entry)

import sleep_edf
from sleep_edf import SleepEDFTrack
import numpy as np
import pandas as pd
import scipy
import sklearn
import mne
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

assert Path(sleep_edf.__file__).resolve().parent == COURSE_DIR / "tracks", (
    "A different sleep_edf module is already loaded. Restart the runtime."
)

runtime_versions = {"python": platform.python_version()}
for distribution in PACKAGE_MAP.values():
    runtime_versions[distribution] = importlib.metadata.version(distribution)

def verificar_runtime_referencia_68():
    diferentes = {
        p: {"F2": reference_runtime[p], "current": runtime_versions[p]}
        for p in ("numpy", "scipy", "scikit-learn", "mne")
        if p in reference_runtime and runtime_versions[p] != reference_runtime[p]
    }
    if diferentes:
        raise RuntimeError("Package versions differ from the F2 reference: " + str(diferentes))

verificar_runtime_referencia_68()
print("Course code revision:", course_revision_loaded)
print("Runtime:", json.dumps(runtime_versions, sort_keys=True))
_SETUP_STAGES_68.add("runtime")


### Setup 9 — original source cell

In [ ]:
# Restore the cache entries in the order recorded in the F2 report.
_require_stage_68("runtime")
resultados_f2_68 = {
    fold: (ref_a_guardada_68 if fold == "A" else _read_json(
        EXPERIMENTS_DIR / f"F2_68_fixed_AE_fold_{fold}.json"
    ))
    for fold in "ABCDE"
}
FOLDS_68 = {k: list(v) for k, v in config_f2_guardada_68["fold_assignment"].items()}
assert FOLDS_68 == EXPECTED_FOLDS_68
DEVELOPMENT_GROUPS_68 = sorted(s for v in FOLDS_68.values() for s in v)
assert len(DEVELOPMENT_GROUPS_68) == len(set(DEVELOPMENT_GROUPS_68)) == 68
assert set(DEVELOPMENT_GROUPS_68).isdisjoint(HOLDOUT_GROUPS)

def identificar_gravacao_reposta_68(entrada):
    if isinstance(entrada, dict):
        entrada = entrada["record"]
    return Path(str(entrada)).name.removesuffix(".edf").removesuffix("-PSG")

registos_guardados_68 = config_f2_guardada_68["recordings"]
ordem_gravacoes_reposta_68 = [
    identificar_gravacao_reposta_68(r) for r in registos_guardados_68
]
assert len(ordem_gravacoes_reposta_68) == len(set(ordem_gravacoes_reposta_68)) == 133

def _listar_caches_b0():
    with os.scandir(CACHE_VIGGY) as entradas:
        return sorted(
            Path(e.path) for e in entradas
            if e.name.startswith("_cache_viggy_b0_features_") and e.name.endswith(".pkl")
        )

caches_b0_para_repor = _drive_io(_listar_caches_b0, "list the B0 caches")
assert len(caches_b0_para_repor) == 133, (
    f"Found {len(caches_b0_para_repor)} B0 caches; expected 133. "
    "Check the folder before continuing. No download will start."
)
cache_por_gravacao_68 = {}
for i, caminho in enumerate(caches_b0_para_repor, 1):
    local = _local_copy(caminho, "features_b0")
    with local.open("rb") as f:
        dados = pickle.load(f)
    assert {"record", "X", "y", "groups", "feature_names"} <= dados.keys()
    record = identificar_gravacao_reposta_68(dados["record"])
    assert record not in cache_por_gravacao_68, f"Duplicate cache for {record}."
    cache_por_gravacao_68[record] = dados
    if i % 25 == 0 or i == len(caches_b0_para_repor):
        print(f"Caches restored: {i}/{len(caches_b0_para_repor)}", flush=True)

assert set(cache_por_gravacao_68) == set(ordem_gravacoes_reposta_68)
b0_gravacoes_68 = [cache_por_gravacao_68[r] for r in ordem_gravacoes_reposta_68]
nomes_features_b0 = list(b0_gravacoes_68[0]["feature_names"])
nomes_features_f2_68 = list(config_f2_guardada_68["feature_names"])
assert len(nomes_features_b0) == 11
assert nomes_features_f2_68[:11] == nomes_features_b0
assert len(nomes_features_f2_68) == 16

_X, _y, _groups, _record_ids = [], [], [], []
for record, dados, registo in zip(
    ordem_gravacoes_reposta_68, b0_gravacoes_68, registos_guardados_68
):
    xx = np.asarray(dados["X"], dtype=float)
    yy = np.asarray(dados["y"]).astype(str)
    gg = np.asarray(dados["groups"]).astype(str)
    if gg.ndim == 0:
        gg = np.full(len(yy), gg.item())
    assert list(dados["feature_names"]) == nomes_features_b0
    assert xx.shape == (len(yy), 11)
    assert gg.shape == yy.shape == (len(yy),)
    assert np.isfinite(xx).all() and np.all(gg == record[:5])
    if isinstance(registo, dict) and "epochs" in registo:
        assert len(yy) == registo["epochs"]
    _X.append(xx); _y.append(yy); _groups.append(gg)
    _record_ids.append(np.full(len(yy), record))

X_b0_68 = np.vstack(_X)
y_b0_68 = np.concatenate(_y)
groups_b0_68 = np.concatenate(_groups)
record_ids_b0_68 = np.concatenate(_record_ids)
del _X, _y, _groups, _record_ids

B0_FEATURE_VERSION = config_f2_guardada_68.get("details", {}).get("base_feature_version", 1)
F2_FEATURE_VERSION = config_f2_guardada_68["feature_version"]
assert B0_FEATURE_VERSION == F2_FEATURE_VERSION == 1
bandas_f2_68 = ["delta", "theta", "alpha", "sigma", "beta"]
indices_bandas_f2_68 = [nomes_features_b0.index(f"eeg_{b}") for b in bandas_f2_68]
potencias_f2_68 = X_b0_68[:, indices_bandas_f2_68]
assert np.all(potencias_f2_68 >= 0)
_total_f2 = potencias_f2_68.sum(axis=1, keepdims=True)
_relativas_f2 = np.divide(
    potencias_f2_68, _total_f2,
    out=np.zeros_like(potencias_f2_68), where=_total_f2 > 0,
)
X_f2_68 = np.column_stack([X_b0_68, _relativas_f2])
assert X_b0_68.shape == (360621, 11)
assert X_f2_68.shape == (360621, 16)
assert np.isfinite(X_f2_68).all()
assert set(groups_b0_68) == set(DEVELOPMENT_GROUPS_68)

SPLITS_68 = {}
previsoes_f2_68 = np.empty_like(y_b0_68)
fold_por_epoca_68 = np.empty(len(y_b0_68), dtype="<U1")
_cobertura_68 = np.zeros(len(y_b0_68), dtype=np.uint8)
for fold in "ABCDE":
    teste = np.isin(groups_b0_68, FOLDS_68[fold])
    tr, te = np.flatnonzero(~teste), np.flatnonzero(teste)
    ref = resultados_f2_68[fold]
    assert ref["fold_assignment"] == FOLDS_68
    np.testing.assert_array_equal(te, ref["test_indices"])
    np.testing.assert_array_equal(y_b0_68[te], ref["report"]["y_true"])
    np.testing.assert_array_equal(groups_b0_68[te], ref["report"]["y_group"])
    assert set(groups_b0_68[tr]).isdisjoint(set(groups_b0_68[te]))
    assert np.all(_cobertura_68[te] == 0)
    assert len(ref["report"]["y_pred"]) == len(te)
    SPLITS_68[fold] = (tr, te)
    previsoes_f2_68[te] = ref["report"]["y_pred"]
    fold_por_epoca_68[te] = fold
    _cobertura_68[te] += 1
assert np.all(_cobertura_68 == 1)

def _erro_walk_68(exc):
    raise exc

def _indexar_psg_68():
    encontrados = {}
    for pasta, _, ficheiros in os.walk(TRAIN_DIR, onerror=_erro_walk_68):
        for nome in ficheiros:
            if nome.endswith("-PSG.edf"):
                record = identificar_gravacao_reposta_68(nome)
                if record in cache_por_gravacao_68:
                    assert record not in encontrados, f"Duplicate PSG: {record}"
                    encontrados[record] = Path(pasta) / nome
    return encontrados

psg_por_gravacao_68 = _drive_io(_indexar_psg_68, "locate the PSG files")
assert set(psg_por_gravacao_68) == set(ordem_gravacoes_reposta_68)
ficheiros_b0_68 = [psg_por_gravacao_68[r] for r in ordem_gravacoes_reposta_68]
limites_gravacoes_68 = np.r_[0, np.cumsum([len(d["y"]) for d in b0_gravacoes_68])]

print("F2 restored:", X_f2_68.shape)
print("Verified 68 participants, 133 recordings and folds A–E.")
_SETUP_STAGES_68.add("features")


### Setup 10 — original source cell

In [ ]:
_require_stage_68("features")
EPOCH_TIMES_VERSION_68 = 1
EPOCH_SECONDS_68 = 30.0
CONSECUTIVE_TOL_68 = 1e-6
RK_TEMPOS_68 = {
    "Sleep stage W": "W", "Sleep stage 1": "N1", "Sleep stage 2": "N2",
    "Sleep stage 3": "N3", "Sleep stage 4": "N3", "Sleep stage R": "REM",
}
META_TEMPOS_68 = {
    "version": EPOCH_TIMES_VERSION_68,
    "course_revision": COURSE_REVISION,
    "epoch_seconds": EPOCH_SECONDS_68,
    "required_native_fs": 100.0,
    "time_origin": "PSG sample zero",
    "event_rounding": True,
    "stage_mapping": RK_TEMPOS_68,
    "mne": mne.__version__,
    "numpy": np.__version__,
}
FICHEIRO_TEMPOS_68 = Path(CACHE_VIGGY) / "viggy_epoch_times_68_v1.json"
_LOCAL_TIMES_PENDING = LOCAL_SESSION_DIR / "epoch_times_pending_v1.json"

def _ler_tempos_edf_68(psg, hyp_local, labels):
    # Read the PSG header and use a small local copy of the hypnogram.
    raw = mne.io.read_raw_edf(psg, preload=False, verbose="ERROR")
    try:
        fs = float(raw.info["sfreq"])
        assert abs(fs - 100.0) < 1e-6 and raw.first_samp == 0
        raw.set_annotations(mne.read_annotations(hyp_local), emit_warning=False)
        eventos, ids = mne.events_from_annotations(
            raw,
            event_id={k: i for i, k in enumerate(RK_TEMPOS_68)},
            use_rounding=True, chunk_duration=EPOCH_SECONDS_68, verbose="ERROR",
        )
        assert np.all(eventos[:, 0] >= 0)
        passo = int(EPOCH_SECONDS_68 * fs)
        eventos = eventos[eventos[:, 0] + passo <= raw.n_times]
        inverso = {codigo: RK_TEMPOS_68[nome] for nome, codigo in ids.items()}
        lidas = np.asarray([inverso[int(e[2])] for e in eventos]).astype(str)
        np.testing.assert_array_equal(lidas, labels, err_msg=f"Labels: {psg.name}")
        return eventos[:, 0].astype(float) / fs, float(raw.n_times / fs)
    finally:
        raw.close()

def _validar_tempos_68(tempos, duracao, labels):
    assert tempos.shape == labels.shape and len(labels) > 0
    assert np.isfinite(tempos).all() and np.isfinite(duracao)
    assert np.all(np.diff(tempos) > 0) and tempos[0] >= 0
    assert tempos[-1] + EPOCH_SECONDS_68 <= duracao + CONSECUTIVE_TOL_68

def preparar_tempos_sessao_68():
    cache = {"metadata": META_TEMPOS_68, "records": {}}
    pendente_reposto = False
    if _drive_is_file(FICHEIRO_TEMPOS_68):
        anterior = _read_json(FICHEIRO_TEMPOS_68)
        if anterior.get("metadata") == META_TEMPOS_68:
            cache = anterior
    if _LOCAL_TIMES_PENDING.is_file():
        pendente = json.loads(_LOCAL_TIMES_PENDING.read_text(encoding="utf-8"))
        if pendente.get("metadata") == META_TEMPOS_68:
            cache["records"].update(pendente["records"])
            pendente_reposto = True

    tempos_por_gravacao, blocos_t, blocos_prev = {}, [], []
    calculadas = reutilizadas = posicao = 0
    alterado = pendente_reposto

    try:
        for numero, (psg, dados) in enumerate(zip(ficheiros_b0_68, b0_gravacoes_68), 1):
            record = identificar_gravacao_reposta_68(dados["record"])
            assert psg.name == f"{record}-PSG.edf"
            assert record[:5] in DEVELOPMENT_GROUPS_68
            labels = np.asarray(dados["y"]).astype(str)

            hyp = psg.with_name(psg.name.replace("-PSG.edf", "-Hypnogram.edf"))
            if not _drive_is_file(hyp):
                candidatos = _drive_io(
                    lambda: sorted(psg.parent.glob(f"{record[:6]}*Hypnogram.edf")),
                    f"locate annotations for {record}",
                )
                assert len(candidatos) == 1, f"Ambiguous annotations: {record}"
                hyp = candidatos[0]

            assinatura = {
                "psg": _file_signature(psg),
                "hypnogram": _file_signature(hyp),
                "labels_sha256": hashlib.sha256("\n".join(labels).encode()).hexdigest(),
            }
            entrada = cache["records"].get(record)
            if entrada is not None and entrada.get("signature") == assinatura:
                tempos = np.asarray(entrada["onsets_seconds"], dtype=float)
                duracao = float(entrada["duration_seconds"])
                reutilizadas += 1
            else:
                hyp_local = _local_copy(hyp, "annotations")
                tempos, duracao = _drive_io(
                    lambda: _ler_tempos_edf_68(psg, hyp_local, labels),
                    f"read the header for {record}",
                )
                _validar_tempos_68(tempos, duracao, labels)
                assert _file_signature(psg) == assinatura["psg"]
                assert _file_signature(hyp) == assinatura["hypnogram"]
                cache["records"][record] = {
                    "signature": assinatura,
                    "onsets_seconds": tempos.tolist(),
                    "duration_seconds": duracao,
                }
                calculadas += 1
                alterado = True

            _validar_tempos_68(tempos, duracao, labels)
            fim = posicao + len(labels)
            assert np.all(record_ids_b0_68[posicao:fim] == record)
            np.testing.assert_array_equal(y_b0_68[posicao:fim], labels)
            posicao = fim

            anterior_valida = np.zeros(len(tempos), dtype=bool)
            anterior_valida[1:] = np.isclose(
                np.diff(tempos), EPOCH_SECONDS_68, rtol=0, atol=CONSECUTIVE_TOL_68
            )
            tempos_por_gravacao[record] = (tempos, duracao)
            blocos_t.append(tempos)
            blocos_prev.append(anterior_valida)

            # Save the first recording, then save at regular checkpoints.
            if numero == 1 or numero % 25 == 0 or numero == len(ficheiros_b0_68):
                if alterado:
                    _atomic_json(FICHEIRO_TEMPOS_68, cache)
                    alterado = False
                if numero % 25 == 0 or numero == len(ficheiros_b0_68):
                    print(f"Epoch times checked: {numero}/{len(ficheiros_b0_68)}", flush=True)
    except (Exception, KeyboardInterrupt):
        # Keep the session work if Drive becomes unavailable.
        _LOCAL_TIMES_PENDING.write_text(
            json.dumps(cache, ensure_ascii=False), encoding="utf-8"
        )
        raise

    assert posicao == len(y_b0_68)
    if _LOCAL_TIMES_PENDING.is_file():
        _LOCAL_TIMES_PENDING.unlink()  # Only the temporary file on local disk.
    return tempos_por_gravacao, np.concatenate(blocos_t), np.concatenate(blocos_prev), calculadas, reutilizadas

(
    tempos_epocas_68, onsets_epocas_68, anterior_consecutiva_68,
    calculados_tempos_68, reutilizados_tempos_68,
) = preparar_tempos_sessao_68()

# Keep the legacy name for compatibility. This does not apply a Wake crop.
tempos_c1_68 = tempos_epocas_68
print(
    f"Epoch times ready: {len(tempos_epocas_68)} recordings | "
    f"{calculados_tempos_68} read now | {reutilizados_tempos_68} reused."
)
print("Epoch-time cache:", FICHEIRO_TEMPOS_68)
_SETUP_STAGES_68.add("times")


### Setup 12 — original source cell

In [ ]:
import copy
import io
import re
import warnings
from datetime import datetime, timezone

_require_stage_68("times")

ESTAGIOS_68 = ["W", "N1", "N2", "N3", "REM"]
METRICAS_68 = ["cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"]

track_ref_68 = SleepEDFTrack(cfg={})
track_b0_68 = track_ref_68
track_c1_68 = track_ref_68
modelo_base_f2_68 = Pipeline([
    ("scale", StandardScaler(**config_f2_guardada_68["scaler_params"])),
    ("clf", RandomForestClassifier(**config_f2_guardada_68["classifier_params"])),
])
modelo_base_b0 = modelo_base_f2_68
relatorio_f2_68 = track_ref_68._make_report(
    y_b0_68, previsoes_f2_68, groups=groups_b0_68, folds=fold_por_epoca_68
)
tabela_f2_68 = pd.DataFrame([
    {"Fold": f, **{m: resultados_f2_68[f]["report"][m] for m in METRICAS_68},
     "Treino (min)": resultados_f2_68[f]["fit_seconds"] / 60}
    for f in "ABCDE"
]).set_index("Fold")
_RESULTADOS_DA_SESSAO_68 = globals().get("_RESULTADOS_DA_SESSAO_68", {})

# Translate display labels without changing saved report fields.
_ENGLISH_DISPLAY_LABELS = {'Treino (min)': 'Fit time (min)', 'Diferença': 'Difference', 'TC1 + corte': 'TC1 + Wake crop', 'Treino antes': 'Training epochs before crop', 'Treino depois': 'Training epochs after crop', 'Wake retiradas': 'Wake epochs removed', 'Teste completo': 'Complete test epochs', 'Estágio': 'Stage', 'Épocas': 'Epochs', 'Recall TC1 + corte': 'Recall TC1 + Wake crop', 'F1 TC1 + corte': 'F1 TC1 + Wake crop', 'Diferença F1': 'F1 change', 'Diferença recall': 'Recall change', 'Diferença kappa': 'Kappa change', 'Kappa TC1': 'TC1 kappa', 'Gravação': 'Recording', 'Com LP40': 'Filtered with LP40', 'Com LP40 (%)': 'Filtered with LP40 (%)', 'Mediana 48–50 (%)': 'Median 48–50 (%)', '48–50 antes (%)': '48–50 before (%)', '48–50 depois (%)': '48–50 after (%)', '0.5–30 mantida (%)': '0.5–30 retained (%)', 'Grupo': 'Group', 'Modelo': 'Model', 'Cumpre critério >=10%': 'Meets the ≥10% criterion', 'Restantes épocas': 'Remaining epochs', 'Recall +LP40': 'Recall + LP40', 'F1 +LP40': 'F1 + LP40'}

def _display_en(value):
    if isinstance(value, pd.DataFrame):
        names = _ENGLISH_DISPLAY_LABELS
        value = value.rename(index=names, columns=names).rename_axis(
            index=lambda name: names.get(name, name),
            columns=lambda name: names.get(name, name),
        )
    display(value)


def assinatura_arrays_68(*arrays):
    h = hashlib.sha256()
    for arr in arrays:
        a = np.asarray(arr)
        if a.dtype.kind in "OUS":
            a = a.astype(str)
        a = np.ascontiguousarray(a)
        h.update(str(a.shape).encode())
        h.update(a.dtype.str.encode())
        h.update(memoryview(a).cast("B"))
    return h.hexdigest()

def juntar_previsoes_68(resultados):
    assert set(resultados) == set("ABCDE")
    previsoes = np.empty_like(y_b0_68)
    cobertura = np.zeros(len(y_b0_68), dtype=np.uint8)
    folds = np.empty(len(y_b0_68), dtype="<U1")
    for fold in "ABCDE":
        r = resultados[fold]
        te = np.asarray(r["test_indices"], dtype=int)
        rep = r["report"]
        np.testing.assert_array_equal(te, SPLITS_68[fold][1])
        np.testing.assert_array_equal(rep["y_true"], y_b0_68[te])
        np.testing.assert_array_equal(rep["y_group"], groups_b0_68[te])
        pred = np.asarray(rep["y_pred"])
        assert pred.shape == te.shape and np.isin(pred, ESTAGIOS_68).all()
        assert np.all(cobertura[te] == 0)
        previsoes[te] = pred
        folds[te] = fold
        cobertura[te] += 1
    assert np.all(cobertura == 1)
    rep = track_ref_68._make_report(
        y_b0_68, previsoes, groups=groups_b0_68, folds=folds
    )
    return previsoes, rep

def avaliar_nova_experiencia_68(nome, X, nomes_features, receita, modelo=None):
    """Perform five fits only when this function is called explicitly."""
    assert SESSAO_PRONTA, "Complete section 1 first."
    verificar_runtime_referencia_68()
    assert re.fullmatch(r"[A-Za-z0-9_-]+", nome), "Use a simple experiment identifier."
    assert nome not in {
        "B0_68_fixed_AE", "F2_68_fixed_AE", "LP40_68_fixed_AE",
        "C1_68_crop30_fixed_AE", "LR1_68_fixed_AE",
    }, "Existing experiment identifiers are reserved."

    X = np.asarray(X)
    assert X.shape == (len(y_b0_68), len(nomes_features))
    assert np.isfinite(X).all()
    base = clone(modelo_base_f2_68 if modelo is None else modelo)
    assert isinstance(base, Pipeline)
    manifesto_modelo = [
        {"step": n, "class": f"{type(e).__module__}.{type(e).__qualname__}",
         "params": e.get_params(deep=False)}
        for n, e in base.steps
    ]
    meta = {
        "experiment": nome, "schema_version": 1,
        "course_revision": COURSE_REVISION,
        "reference_experiment": "F2_68_fixed_AE",
        "split_unit": "subject", "cv_method": "five fixed subject folds A-E",
        "fold_assignment": FOLDS_68, "n_subjects": 68,
        "n_recordings": 133, "n_epochs": len(y_b0_68),
        "feature_names": list(nomes_features), "recipe": receita,
        "training_selection": "all original retained training epochs",
        "data_sha256": assinatura_arrays_68(X, y_b0_68, groups_b0_68, record_ids_b0_68),
        "model": manifesto_modelo, "runtime": runtime_versions,
        "feature_cache_runtime": reference_runtime,
    }
    meta = json.loads(json.dumps(meta, default=converter_numpy))
    resultados = {}
    for fold in "ABCDE":
        tr, te = SPLITS_68[fold]
        assert set(groups_b0_68[te]) == set(FOLDS_68[fold])
        assert set(groups_b0_68[tr]).isdisjoint(set(groups_b0_68[te]))
        destino = EXPERIMENTS_DIR / f"{nome}_fold_{fold}.json"
        novo = False
        if _drive_is_file(destino):
            r = _read_json(destino)
        elif (nome, fold) in _RESULTADOS_DA_SESSAO_68:
            r = _RESULTADOS_DA_SESSAO_68[(nome, fold)]
            novo = True
        else:
            print(f"{nome} — fold {fold}: fitting...", flush=True)
            clf = clone(base)
            inicio = time.perf_counter()
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                clf.fit(X[tr], y_b0_68[tr])
            fit_seconds = time.perf_counter() - inicio
            estimator = clf.steps[-1][1]
            n_iter = getattr(estimator, "n_iter_", None)
            if n_iter is not None and hasattr(estimator, "max_iter"):
                assert np.all(np.asarray(n_iter) < estimator.max_iter), (
                    "The fit reached max_iter. Resolve this before accepting the result."
                )
            inicio = time.perf_counter()
            pred = clf.predict(X[te])
            predict_seconds = time.perf_counter() - inicio
            r = {
                "metadata": copy.deepcopy(meta), "fold": fold,
                "created_at_utc": datetime.now(timezone.utc).isoformat(),
                "train_indices": tr, "test_indices": te,
                "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
                "n_iter": n_iter, "converged": True,
                "report": track_ref_68._make_report(
                    y_b0_68[te], pred, groups=groups_b0_68[te],
                    folds=np.full(len(te), fold),
                ),
            }
            novo = True
        assert r["metadata"] == meta, (
            f"{nome}, fold {fold}: different configuration or data. "
            "Use a new identifier or version. The previous file has been preserved."
        )
        assert r["fold"] == fold and r["converged"]
        np.testing.assert_array_equal(r["train_indices"], tr)
        np.testing.assert_array_equal(r["test_indices"], te)
        np.testing.assert_array_equal(r["report"]["y_true"], y_b0_68[te])
        np.testing.assert_array_equal(r["report"]["y_group"], groups_b0_68[te])
        assert len(r["report"]["y_pred"]) == len(te)
        _RESULTADOS_DA_SESSAO_68[(nome, fold)] = r
        if novo:
            _atomic_json(destino, r)
        resultados[fold] = r
        print(
            f"Fold {fold}: kappa {r['report']['cohens_kappa']:.3f} | "
            f"fit {r['fit_seconds']/60:.2f} min"
            + ("" if novo else " | reused"),
            flush=True,
        )
    tabela = pd.DataFrame([
        {"Fold": f, **{m: resultados[f]["report"][m] for m in METRICAS_68},
         "Treino (min)": resultados[f]["fit_seconds"] / 60}
        for f in "ABCDE"
    ]).set_index("Fold")
    previsoes, relatorio = juntar_previsoes_68(resultados)
    return resultados, tabela, previsoes, relatorio

def apresentar_experiencia_68(nome, resultados, tabela, previsoes, relatorio, guardar=True):
    """Display and export the results without fitting again."""
    print("METRICS BY FOLD")
    _display_en(tabela.round(3))
    print(
        f"Mean fold kappa ± SD: {tabela['cohens_kappa'].mean():.3f}"
        f" ± {tabela['cohens_kappa'].std(ddof=1):.3f}"
    )
    print(f"Total fit time: {tabela['Treino (min)'].sum():.1f} min")
    comparacao = pd.DataFrame({
        "F2": {m: relatorio_f2_68[m] for m in METRICAS_68},
        nome: {m: relatorio[m] for m in METRICAS_68},
    })
    comparacao["Diferença"] = comparacao[nome] - comparacao["F2"]
    print("\nPOOLED METRICS")
    _display_en(comparacao.round(3))
    print("\nRESULTS BY STAGE")
    print(classification_report(
        y_b0_68, previsoes, labels=ESTAGIOS_68, digits=3, zero_division=0
    ))
    classes = classification_report(
        y_b0_68, previsoes, labels=ESTAGIOS_68, output_dict=True, zero_division=0
    )
    sujeito = pd.DataFrame({
        "F2": pd.DataFrame(relatorio_f2_68["per_group"]).set_index("group")["cohens_kappa"],
        nome: pd.DataFrame(relatorio["per_group"]).set_index("group")["cohens_kappa"],
    })
    assert len(sujeito) == 68 and not sujeito.isna().any().any()
    sujeito["Diferença"] = sujeito[nome] - sujeito["F2"]
    dif = sujeito["Diferença"]
    print(
        "\nParticipants — kappa before rounding:",
        f"{int((dif > 0).sum())} improved |",
        f"{int((dif < 0).sum())} decreased |",
        f"{int((dif == 0).sum())} tied",
    )
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, pred, titulo in [
        (axes[0], previsoes_f2_68, "F2 — reference"),
        (axes[1], previsoes, nome),
    ]:
        disp = ConfusionMatrixDisplay.from_predictions(
            y_b0_68, pred, labels=ESTAGIOS_68, normalize="true",
            values_format=".3f", cmap="Blues", colorbar=False, ax=ax,
        )
        disp.im_.set_clim(0, 1)
        ax.set_title(titulo)
    fig.suptitle("68 subjects — fixed folds A–E — complete test recordings")
    fig.tight_layout(rect=(0, 0, 1, .94))
    plt.show()
    plt.close(fig)

    if guardar:
        exportacao = {
            **resultados["A"]["metadata"],
            "cv_mean_sd_across_folds": {
                m: {"mean": float(tabela[m].mean()), "sd": float(tabela[m].std(ddof=1))}
                for m in METRICAS_68
            },
            "sd_ddof": 1, "fold_metrics": tabela.reset_index().to_dict(orient="records"),
            "total_fit_seconds": sum(float(r["fit_seconds"]) for r in resultados.values()),
            "total_predict_seconds": sum(float(r["predict_seconds"]) for r in resultados.values()),
            "report": relatorio, "classification_report": classes,
            "comparison_vs_F2_global": comparacao.rename_axis("metric").reset_index().to_dict(orient="records"),
            "comparison_vs_F2_subject_kappa": sujeito.reset_index().to_dict(orient="records"),
            "working_decision": "Pending review; no automatic replacement of F2.",
        }
        destino = EXPERIMENTS_DIR / f"{nome}.json"
        if not _drive_is_file(destino):
            _atomic_json(destino, exportacao)
            print("Report saved:", destino)
        else:
            print("Existing report preserved:", destino)
        imagem = EXPERIMENTS_DIR / f"{nome}_confusion.png"
        if _drive_is_file(imagem):
            print("Existing image preserved:", imagem)
            return comparacao, sujeito
        fig_ind, ax = plt.subplots(figsize=(7, 6))
        disp = ConfusionMatrixDisplay.from_predictions(
            y_b0_68, previsoes, labels=ESTAGIOS_68, normalize="true",
            values_format=".3f", cmap="Blues", colorbar=False, ax=ax,
        )
        disp.im_.set_clim(0, 1)
        ax.set_title(f"{nome} — 68 subjects — fixed folds A–E")
        fig_ind.tight_layout()
        buffer = io.BytesIO()
        fig_ind.savefig(buffer, format="png", dpi=200, bbox_inches="tight")
        plt.close(fig_ind)
        _atomic_bytes(imagem, buffer.getvalue())
        print("Image saved:", imagem)
    return comparacao, sujeito

_SETUP_STAGES_68.add("helpers")


### Setup 13 — original source cell

In [ ]:
# Build context with fixed operations per recording; no fitting.
_require_stage_68("helpers")
TC1_FEATURE_VERSION = 1
NOME_TC1_68 = "TC1_68_fixed_AE"
TC1_CONFIG_68 = {
    "version": TC1_FEATURE_VERSION,
    "base_features": "F2 version 1, original 11 plus five relative EEG powers",
    "previous_epochs": 1,
    "direction": "past only",
    "epoch_seconds": EPOCH_SECONDS_68,
    "consecutive_tolerance_seconds": CONSECUTIVE_TOL_68,
    "require_same_recording": True,
    "missing_previous": "copy current features and set availability flag to zero",
    "availability_flag": True,
    "additional_filter": None,
    "training_selection": "all retained epochs",
    "label_inputs": False,
}

def preparar_contexto_anterior_68(X_atual=None):
    X_atual = X_f2_68 if X_atual is None else np.asarray(X_atual)
    assert X_atual.shape == X_f2_68.shape
    validas = np.asarray(anterior_consecutiva_68, dtype=bool)
    assert validas.shape == (len(X_atual),) and not validas[0]
    indices = np.flatnonzero(validas)
    assert np.all(record_ids_b0_68[indices] == record_ids_b0_68[indices - 1])
    np.testing.assert_allclose(
        onsets_epocas_68[indices] - onsets_epocas_68[indices - 1],
        EPOCH_SECONDS_68, rtol=0, atol=CONSECUTIVE_TOL_68,
    )
    anteriores = X_atual.copy()
    anteriores[indices] = X_atual[indices - 1]
    X_contexto = np.column_stack([X_atual, anteriores, validas.astype(float)])
    nomes = (
        list(nomes_features_f2_68)
        + [f"previous_{n}" for n in nomes_features_f2_68]
        + ["previous_epoch_available"]
    )
    assert X_contexto.shape == (len(y_b0_68), 33)
    assert np.isfinite(X_contexto).all()
    np.testing.assert_array_equal(X_contexto[:, :16], X_atual)
    np.testing.assert_array_equal(X_contexto[indices, 16:32], X_atual[indices - 1])
    return X_contexto, nomes



for _stage in ("drive", "runtime", "features", "times", "helpers"):
    _require_stage_68(_stage)
assert X_b0_68.shape == (360621, 11)
assert X_f2_68.shape == (360621, 16)
assert len(onsets_epocas_68) == len(anterior_consecutiva_68) == len(y_b0_68)
assert len(tempos_epocas_68) == len(ficheiros_b0_68) == 133
assert len(set(groups_b0_68)) == 68
assert FOLDS_68 == EXPECTED_FOLDS_68
assert not set(groups_b0_68).intersection(HOLDOUT_GROUPS)
assert modelo_base_f2_68.named_steps["clf"].n_estimators == 200
assert modelo_base_f2_68.named_steps["clf"].class_weight == "balanced"
assert modelo_base_f2_68.named_steps["clf"].random_state == 0
verificar_runtime_referencia_68()

X_tc1_68, nomes_features_tc1_68 = preparar_contexto_anterior_68()
track = trk = track_ref_68
SESSAO_PRONTA = True

print("SESSION READY")
print(f"F2: {X_f2_68.shape} | TC1: {X_tc1_68.shape}")
print("68 participants | 133 recordings | folds A–E")
print("Epochs with a consecutive predecessor:", int(anterior_consecutiva_68.sum()))


## 2. Verify existing YASA predictions (default, no training)
Reads saved A–E predictions and compares with the TC1+C1 baseline. Checks coverage, fold indices and metrics; saves a separate verification CSV without overwriting the original results.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score, balanced_accuracy_score, confusion_matrix

assert SESSAO_PRONTA and X_tc1_68.shape == (360621,33)
YASA_OUT = Path('/content/drive/MyDrive/CM2013_Dalya_experiments/v5_yasa_spindles_AE')
VERSION = 'yasa_spindles_FpzCz_v1_count_duration_amp_frequency'
PRED_DIR = YASA_OUT / 'fold_predictions'
truth = np.asarray(y_b0_68).astype(str)
labels = list(ESTAGIOS_68)
pred = np.empty(len(truth), dtype='<U8')
seen = np.zeros(len(truth), dtype=bool)
fold_stats = []
for fold in 'ABCDE':
    path = PRED_DIR / f'{VERSION}_fold_{fold}.npz'
    if not path.exists():
        raise FileNotFoundError(f'Missing saved fold {fold}: {path}. No training will be started.')
    with np.load(path, allow_pickle=False) as z:
        idx = np.asarray(z['test_indices'],dtype=int)
        yhat = np.asarray(z['pred']).astype(str)
    expected = np.asarray(SPLITS_68[fold][1], dtype=int)
    np.testing.assert_array_equal(idx, expected)
    assert len(yhat)==len(idx) and np.isin(yhat, labels).all()
    assert not seen[idx].any(), f'Overlapping fold {fold}'
    pred[idx] = yhat
    seen[idx] = True
    fold_stats.append({'fold':fold,'kappa':cohen_kappa_score(truth[idx],yhat,labels=labels),
                       'n1_recall':recall_score(truth[idx],yhat,labels=['N1'],average='macro',zero_division=0)})
assert seen.all(), f'Missing {np.count_nonzero(~seen)} epochs'
# Load the saved team baseline directly; validate alignment and fold assignment.
BASELINE_NAME = 'TC1_C1_68_crop30_fixed_AE'
BASELINE_FILE = EXPERIMENTS_DIR / f'{BASELINE_NAME}.json'
if not _drive_is_file(BASELINE_FILE):
    raise FileNotFoundError(f'Missing baseline JSON: {BASELINE_FILE}')
baseline_saved = _read_json(BASELINE_FILE)
assert baseline_saved['experiment'] == BASELINE_NAME
assert baseline_saved['course_revision'] == COURSE_REVISION
assert baseline_saved['fold_assignment'] == FOLDS_68
baseline_report = baseline_saved['report']
assert baseline_report['labels'] == labels
np.testing.assert_array_equal(baseline_report['y_true'], truth)
np.testing.assert_array_equal(baseline_report['y_group'], groups_b0_68)
np.testing.assert_array_equal(baseline_report['y_fold'], fold_por_epoca_68)
base = np.asarray(baseline_report['y_pred']).astype(str)
assert base.shape == truth.shape and np.isin(base, labels).all()

assert base is not None and len(base)==len(truth), 'Missing saved team baseline predictions'
base = np.asarray(base).astype(str)
def evaluate(p):
    return {'accuracy':accuracy_score(truth,p), 'kappa':cohen_kappa_score(truth,p,labels=labels),
            'macro_f1':f1_score(truth,p,labels=labels,average='macro',zero_division=0),
            'balanced_accuracy':balanced_accuracy_score(truth,p),
            'n1_recall':recall_score(truth,p,labels=['N1'],average='macro',zero_division=0),
            'n1_precision':precision_score(truth,p,labels=['N1'],average='macro',zero_division=0),
            'n1_f1':f1_score(truth,p,labels=['N1'],average='macro',zero_division=0)}
summary = pd.DataFrame([{'experiment':'TC1+C1 baseline',**evaluate(base)},
                        {'experiment':'TC1+C1 + YASA',**evaluate(pred)}])
print('YASA SAVED PREDICTIONS VERIFIED — 5 folds,',len(truth),'epochs')
display(summary)
display(pd.DataFrame(fold_stats))
expected = {'accuracy':0.878188,'kappa':0.748767,'macro_f1':0.687821,'n1_recall':0.227499}
for key,val in expected.items():
    assert abs(summary.iloc[1][key]-val)<0.00001, f'{key}: got {summary.iloc[1][key]:.8f}, expected {val}'
print('Original YASA global metrics reproduced (tolerance 1e-5).')


## 3. Optional comparison figure and documentation
Generated files go to `v5_yasa_spindles_AE/clean_verification/`; original experiment outputs remain unchanged.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
outdoc = YASA_OUT/'clean_verification'
outdoc.mkdir(parents=True, exist_ok=True)
summary.to_csv(outdoc/'baseline_vs_yasa_verified.csv',index=False)
pd.DataFrame(fold_stats).to_csv(outdoc/'folds_verified.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, title, p in zip(axes,['TC1+C1 baseline','TC1+C1 + YASA'],[base,pred]):
    cm=confusion_matrix(truth,p,labels=labels)
    normalized=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    im=ax.imshow(normalized,vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(len(labels)),labels);ax.set_yticks(range(len(labels)),labels)
    ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.set_title(title)
    for (r,c),value in np.ndenumerate(normalized):
        ax.text(c,r,f'{value:.2f}',ha='center',va='center',fontsize=9,color='white' if value>0.55 else 'black')
    pd.DataFrame(cm,index=labels,columns=labels).to_csv(outdoc/(title.replace(' ','_').replace('+','plus')+'_confusion_counts.csv'))
fig.colorbar(im,ax=axes,orientation='horizontal',fraction=0.06,pad=0.12,label='Row-normalized fraction')
fig.savefig(outdoc/'baseline_vs_yasa_confusion.png',dpi=180,bbox_inches='tight')
plt.show()
print('Saved verification files to',outdoc)


## 4. Optional original YASA full experiment — **disabled by default**
This cell contains the original v5 feature extraction, checkpoint reuse and model fitting logic. Enable only if you deliberately want to reproduce the full pipeline. **Missing checkpoints will trigger training when enabled.** Never enable for routine documentation.

In [ ]:
RUN_FULL_YASA_PIPELINE = False  # Explicit opt-in only; may extract EEG and train missing folds.
if RUN_FULL_YASA_PIPELINE:
    # Full experiment — YASA spindle features + original TC1+C1, folds A–E
    import sys, subprocess, importlib.util, json, hashlib, time
    from pathlib import Path
    import numpy as np, pandas as pd
    from sklearn.base import clone
    from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, recall_score, precision_score

    assert SESSAO_PRONTA and X_tc1_68.shape == (360621, 33)
    if importlib.util.find_spec('yasa') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'yasa'])
    import yasa, mne

    OUT = Path('/content/drive/MyDrive/CM2013_Dalya_experiments/v5_yasa_spindles_AE')
    FCACHE = OUT / 'feature_cache'; PCACHE = OUT / 'fold_predictions'
    FCACHE.mkdir(parents=True, exist_ok=True); PCACHE.mkdir(parents=True, exist_ok=True)
    VERSION = 'yasa_spindles_FpzCz_v1_count_duration_amp_frequency'
    SPINDLE_FEATURES = ['yasa_sp_count', 'yasa_sp_mean_duration', 'yasa_sp_mean_amplitude', 'yasa_sp_mean_frequency']
    record_ids = np.asarray(record_ids_b0_68, dtype=str)
    onsets = np.asarray(onsets_epocas_68, dtype=float)
    assert len(record_ids) == len(onsets) == len(y_b0_68)
    assert set(record_ids) == set(ordem_gravacoes_reposta_68)
    assert set(DEVELOPMENT_GROUPS_68).isdisjoint(HOLDOUT_GROUPS)

    # Use spindle event midpoint to assign each detected event to ONE 30s epoch.
    # Empty-epoch features are zero; mean amplitude/duration/frequency are zero if no event.
    def extract_spindles(record, ix):
        path = _local_copy(psg_por_gravacao_68[record], 'v5_yasa_psg')
        raw = mne.io.read_raw_edf(str(path), preload=False, verbose='ERROR')
        try:
            ch = next((c for c in raw.ch_names if 'EEG Fpz-Cz' in c), None)
            if ch is None: raise ValueError(f'EEG Fpz-Cz missing: {record}')
            sf = float(raw.info['sfreq'])
            eeg_uv = raw.get_data(picks=[ch])[0] * 1e6
        finally:
            raw.close()
        if not np.isfinite(eeg_uv).all(): raise ValueError(f'Non-finite EEG: {record}')
        starts = onsets[ix]
        if np.any(starts < 0) or np.any(starts + 30 > len(eeg_uv)/sf + 1/sf):
            raise ValueError(f'Invalid epoch alignment: {record}')
        if np.any(np.diff(starts) <= 0): raise ValueError(f'Unsorted onsets: {record}')
        detection = yasa.spindles_detect(eeg_uv, sf=sf, ch_names=[ch], verbose=False)
        events = detection.summary() if detection is not None else pd.DataFrame()
        result = np.zeros((len(ix), 4), dtype=np.float32)
        if len(events):
            for required in ('Start','End','Duration','Amplitude','Frequency'):
                if required not in events.columns: raise ValueError(f'Missing YASA field {required}: {list(events.columns)}')
            mid = (events['Start'].to_numpy(float) + events['End'].to_numpy(float)) / 2
            # Search in actual retained epoch starts; omitted epochs are not assigned.
            slot = np.searchsorted(starts, mid, side='right') - 1
            valid = (slot >= 0) & (slot < len(starts))
            valid &= np.where(valid, mid < starts[np.clip(slot, 0, len(starts)-1)] + 30, False)
            for j in np.flatnonzero(valid):
                k = int(slot[j]); result[k,0] += 1
                result[k,1] += float(events.iloc[j]['Duration'])
                result[k,2] += float(events.iloc[j]['Amplitude'])
                result[k,3] += float(events.iloc[j]['Frequency'])
            nonzero = result[:,0] > 0
            result[nonzero,1:] /= result[nonzero,0,None]
        if not np.isfinite(result).all(): raise ValueError(f'Invalid features: {record}')
        return result, len(events)

    X_sp = np.zeros((len(y_b0_68), 4), dtype=np.float32)
    record_summary = []
    for i, record in enumerate(ordem_gravacoes_reposta_68, 1):
        ix = np.flatnonzero(record_ids == record)
        assert len(ix) and np.all(np.diff(ix) == 1)
        save = FCACHE / f'{record}_{VERSION}.npz'
        if save.exists():
            with np.load(save, allow_pickle=False) as z:
                np.testing.assert_array_equal(z['onsets'], onsets[ix])
                values = z['features'].copy(); n_events = int(z['n_events'])
            assert values.shape == (len(ix), 4)
        else:
            values, n_events = extract_spindles(record, ix)
            temp = save.with_name(save.stem + '.partial.npz')
            np.savez_compressed(temp, onsets=onsets[ix], features=values, n_events=n_events)
            temp.replace(save)
        X_sp[ix] = values
        record_summary.append({'record':record, 'epochs':len(ix), 'spindles_detected':n_events, 'epochs_with_spindle':int(np.count_nonzero(values[:,0]))})
        if i % 10 == 0 or i == 133: print(f'YASA features saved: {i}/133', flush=True)
    assert np.isfinite(X_sp).all()
    pd.DataFrame(record_summary).to_csv(OUT/'record_detection_summary.csv', index=False)
    X_new = np.column_stack([X_tc1_68, X_sp])
    assert X_new.shape == (360621, 37)

    # Original C1 crop indices, not recalculated using held-out labels.
    crop = {}
    for fold in 'ABCDE':
        r = _read_json(EXPERIMENTS_DIR / f'C1_68_crop30_fixed_AE_fold_{fold}.json')
        tr = np.asarray(r['train_indices'], dtype=int)
        original_train, te = SPLITS_68[fold]
        np.testing.assert_array_equal(np.asarray(r['test_indices'], dtype=int), te)
        assert np.isin(tr, original_train).all() and np.intersect1d(tr, te).size == 0
        assert np.all(np.asarray(y_b0_68)[np.setdiff1d(original_train, tr)] == 'W')
        crop[fold] = (tr, te)

    config = {'version':VERSION,'feature_names':SPINDLE_FEATURES,'model':'TC1+C1 + YASA spindles',
              'subject_folds':'ABCDE','feature_columns':37,'assignment':'event midpoint in retained epoch',
              'original_crop':'C1_68_crop30_fixed_AE','n_recordings':133}
    (OUT/'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    all_pred = np.empty(len(y_b0_68), dtype='<U4'); seen = np.zeros(len(y_b0_68), bool)
    fold_rows = []
    for fold in 'ABCDE':
        tr, te = crop[fold]
        save = PCACHE / f'{VERSION}_fold_{fold}.npz'
        if save.exists():
            with np.load(save, allow_pickle=False) as z:
                np.testing.assert_array_equal(z['test_indices'], te)
                pred = z['pred'].astype(str)
            print('Fold', fold, 'reused checkpoint', flush=True)
        else:
            print('Fold', fold, 'training...', flush=True)
            model = clone(modelo_base_f2_68)
            t0 = time.perf_counter()
            model.fit(X_new[tr], np.asarray(y_b0_68)[tr])
            pred = np.asarray(model.predict(X_new[te]), dtype=str)
            elapsed = time.perf_counter()-t0
            temp = save.with_name(save.stem + '.partial.npz')
            np.savez_compressed(temp, test_indices=te, pred=pred, fit_predict_seconds=elapsed)
            temp.replace(save)
            print('Fold', fold, 'saved', flush=True)
        assert len(pred) == len(te) and np.isin(pred, ESTAGIOS_68).all()
        assert not seen[te].any()
        all_pred[te] = pred; seen[te] = True
        truth = np.asarray(y_b0_68)[te]
        fold_rows.append({'fold':fold,'kappa':cohen_kappa_score(truth,pred,labels=ESTAGIOS_68),
           'macro_f1':f1_score(truth,pred,labels=ESTAGIOS_68,average='macro',zero_division=0),
           'n1_recall':recall_score(truth,pred,labels=['N1'],average='macro',zero_division=0),
           'n1_f1':f1_score(truth,pred,labels=['N1'],average='macro',zero_division=0),
           'n2_f1':f1_score(truth,pred,labels=['N2'],average='macro',zero_division=0)})
        pd.DataFrame(fold_rows).to_csv(OUT/'fold_metrics.csv', index=False)
    assert seen.all()

    def metrics(p):
        y = np.asarray(y_b0_68)
        return {'kappa':cohen_kappa_score(y,p,labels=ESTAGIOS_68),
           'macro_f1':f1_score(y,p,labels=ESTAGIOS_68,average='macro',zero_division=0),
           'accuracy':accuracy_score(y,p),
           'n1_recall':recall_score(y,p,labels=['N1'],average='macro',zero_division=0),
           'n1_precision':precision_score(y,p,labels=['N1'],average='macro',zero_division=0),
           'n1_f1':f1_score(y,p,labels=['N1'],average='macro',zero_division=0),
           'n2_f1':f1_score(y,p,labels=['N2'],average='macro',zero_division=0)}
    base = np.asarray(_read_json(EXPERIMENTS_DIR / 'TC1_C1_68_crop30_fixed_AE.json')['report']['y_pred'])
    assert base is not None and len(base) == len(all_pred)
    rows = [{'experiment':'TC1+C1 reference',**metrics(np.asarray(base))},
            {'experiment':'TC1+C1 + YASA spindles',**metrics(all_pred)}]
    comparison = pd.DataFrame(rows)
    comparison.to_csv(OUT/'comparison.csv',index=False)
    lines = ['# TC1+C1 + YASA spindle features — A–E evaluation', '',
     '68 development subjects; 133 PSG recordings; 10 holdout subjects untouched.',
     'Same fixed subject folds and C1 training-only Wake crop as baseline.',
     'Four event-based YASA spindle features; no K-complex features.',
     'YASA detections are not expert-verified. Features are computed from continuous EEG without stage labels.',
     'These results are exploratory until independently reviewed; no holdout was used.', '',
     comparison.to_markdown(index=False, floatfmt='.4f'), '',
     '## Per-fold results', '', pd.DataFrame(fold_rows).to_markdown(index=False,floatfmt='.4f'), '']
    (OUT/'RESULTS_YASA_AE.md').write_text('\n'.join(lines),encoding='utf-8')
    print('ALL FOLDS FINISHED — results:', OUT)
    display(comparison)
    display(pd.DataFrame(fold_rows))
